In [4]:
!pip install bertopic Sastrawi pandas plotly


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import plotly.io as pio
pio.renderers.default = "notebook"

In [6]:
# Load dataset
df = pd.read_csv('dataset_berita_merge.csv')

# Cek missing values
print("=== INFO DATASET ===")
print(f"Total dokumen        : {len(df)}")
print(f"Missing values (isi) : {df['isi'].isna().sum()}")

# Hapus baris kosong
df = df.dropna(subset=['isi'])
docs = df['isi'].tolist()

# Statistik panjang teks
panjang = [len(d.split()) for d in docs]
print(f"\n=== STATISTIK TEKS ===")
print(f"Rata-rata panjang : {sum(panjang)/len(panjang):.1f} kata")
print(f"Terpendek         : {min(panjang)} kata")
print(f"Terpanjang        : {max(panjang)} kata")

# Preview 3 dokumen pertama
print("\n=== CONTOH DOKUMEN ===")
for i, doc in enumerate(docs[:3]):
    print(f"\n[{i+1}] {doc[:200]}...")

=== INFO DATASET ===
Total dokumen        : 2238
Missing values (isi) : 0

=== STATISTIK TEKS ===
Rata-rata panjang : 287.4 kata
Terpendek         : 16 kata
Terpanjang        : 2339 kata

=== CONTOH DOKUMEN ===

[1] klarifikasi video tki histeris lihat ledak rudal hasil rekayasa ai klarifikasi dasar verifikasi kompascom jauh perlu lurus kait informasi kompascom media sosial edar video narasi orang tenaga kerja pe...

[2] geely cepat inovasi bas ai hadir mobilitas aman cerdas beijing kompascom pamer beijing auto show geely tampil teknologi energi baru smart cities smart energy smart platforms ekosistem metanol masuk da...

[3] google siap investasi hingga rp triliun anthropic kompascom google lalu induk alphabet kabar gelontor investasi hingga miliar dollar as rp triliun usaha cerdas buat anthropic informasi pertama kali la...


In [7]:
# Ambil stopwords dasar Bahasa Indonesia dari Sastrawi
factory = StopWordRemoverFactory()
base_stopwords = factory.get_stop_words()

# Custom stopwords tambahan — kata-kata noise spesifik di dataset berita ini
custom_stopwords = [
    'kompascom', 'baca', 'video', 'foto', 'nya', 'yg',
    'dan', 'di', 'dari', 'ke', 'ini', 'itu', 'yang', 'dengan',
    'untuk', 'pada', 'juga', 'sudah', 'akan', 'ada', 'bisa',
    'kata', 'ujar', 'ungkap', 'sebut', 'tutur', 'jelas'
]

stopword_list = list(set(base_stopwords + custom_stopwords))

print(f"Total stopwords digunakan: {len(stopword_list)}")
print(f"Contoh stopwords custom  : {custom_stopwords[:10]}")

# Inisiasi CountVectorizer dengan stopwords Indonesia
vectorizer_model = CountVectorizer(
    stop_words=stopword_list,
    min_df=5,          # Abaikan kata yang muncul < 5 dokumen (mengurangi noise)
    max_df=0.85,       # Abaikan kata yang muncul di > 85% dokumen (terlalu umum)
    ngram_range=(1, 2) # Tangkap juga bigram seperti "presiden jokowi", "bank indonesia"
)

Total stopwords digunakan: 135
Contoh stopwords custom  : ['kompascom', 'baca', 'video', 'foto', 'nya', 'yg', 'dan', 'di', 'dari', 'ke']


In [8]:
# Inisiasi BERTopic
# - language="multilingual" → pakai embedding yang mengerti Bahasa Indonesia
# - verbose=True → tampilkan log progress training
topic_model = BERTopic(
    language="multilingual",
    vectorizer_model=vectorizer_model,
    verbose=True,
    min_topic_size=15,       # Minimum 15 dokumen per topik (hindari topik terlalu kecil)
    nr_topics="auto",        # Biarkan model tentukan jumlah topik optimal otomatis
    calculate_probabilities=False  # False = lebih cepat, cukup untuk analisis topik
)

# Training — proses ini memakan waktu beberapa menit
print("Memulai training BERTopic...")
topics, probs = topic_model.fit_transform(docs)
print(f"\nTraining selesai! Ditemukan {len(set(topics)) - 1} topik (di luar outlier).")

2026-04-26 12:08:32,334 - BERTopic - Embedding - Transforming documents to embeddings.


Memulai training BERTopic...


Batches:   0%|          | 0/70 [00:00<?, ?it/s]

2026-04-26 12:11:09,048 - BERTopic - Embedding - Completed ✓
2026-04-26 12:11:09,054 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-26 12:11:21,500 - BERTopic - Dimensionality - Completed ✓
2026-04-26 12:11:21,508 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-26 12:11:21,717 - BERTopic - Cluster - Completed ✓
2026-04-26 12:11:21,719 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-04-26 12:11:24,759 - BERTopic - Representation - Completed ✓
2026-04-26 12:11:24,763 - BERTopic - Topic reduction - Reducing number of topics
2026-04-26 12:11:24,849 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-26 12:11:27,552 - BERTopic - Representation - Completed ✓
2026-04-26 12:11:27,561 - BERTopic - Topic reduction - Reduced number of topics from 33 to 6



Training selesai! Ditemukan 5 topik (di luar outlier).


In [9]:
# Tampilkan ringkasan semua topik
topic_info = topic_model.get_topic_info()

print("=== RINGKASAN TOPIK ===")
print(topic_info.to_string())

# Tampilkan detail kata kunci per topik (top 5 topik terbesar)
print("\n=== DETAIL KATA KUNCI PER TOPIK ===")
for topic_id in topic_info['Topic'].values[1:6]:  # Skip -1 (outlier)
    words = topic_model.get_topic(topic_id)
    kata = [w[0] for w in words[:8]]
    print(f"\nTopik {topic_id:>2} | Kata kunci: {', '.join(kata)}")

=== RINGKASAN TOPIK ===
   Topic  Count                            Name                                                                   Representation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [10]:
# Bar chart kata kunci dengan skor c-TF-IDF per topik
# c-TF-IDF = seberapa penting kata tersebut dalam topik ini dibanding topik lain
fig_barchart = topic_model.visualize_barchart(
    top_n_topics=10,
    n_words=10,
    title="<b>Top Kata Kunci per Topik Berita</b>"
)
fig_barchart.show()

In [11]:
# Peta jarak antar topik dalam ruang 2D (menggunakan reduksi dimensi UMAP)
# Topik yang berdekatan = topik yang semantiknya mirip
# Ukuran lingkaran = jumlah dokumen dalam topik tersebut
fig_intertopic = topic_model.visualize_topics(
    title="<b>Intertopic Distance Map — Berita Indonesia</b>"
)
fig_intertopic.show()

In [12]:
# Dendrogram — menunjukkan bagaimana topik-topik bisa digabungkan secara hierarkis
# Berguna untuk melihat apakah ada topik-topik yang sebetulnya "serumpun"
fig_hierarchy = topic_model.visualize_hierarchy(
    title="<b>Hierarchical Clustering Topik Berita</b>"
)
fig_hierarchy.show()

In [13]:
# Heatmap korelasi antar topik berdasarkan distribusi kata
# Warna lebih terang = dua topik lebih mirip satu sama lain
fig_heatmap = topic_model.visualize_heatmap(
    title="<b>Topic Similarity Heatmap</b>"
)
fig_heatmap.show()

In [14]:
# Sebaran seluruh dokumen berita dalam ruang 2D, diwarnai berdasarkan topiknya
# Ini visualisasi paling berat — bisa lambat jika RAM terbatas
# Jika crash, cukup skip cell ini
fig_documents = topic_model.visualize_documents(
    docs,
    title="<b>Sebaran Dokumen Berita per Topik</b>"
)
fig_documents.show()

In [17]:
# =====================================================================
# INTERPRETASI TOPIK — ISI SESUAI HASIL TRAINING DI MESINMU
# Setelah model selesai training, cek get_topic_info() dan isi label
# di bawah ini berdasarkan kata kunci yang muncul.
# =====================================================================

interpretasi = {
    # Contoh format — GANTI sesuai hasil aktualmu:
    0:  "ai, ekonomi, teknologi, nilai, rp, persen, sekolah, sistem",   # misal kata kunci: presiden, menteri, pemerintah
    1:  "eropa, menang, jerman, prancis, italia, final, grup, latih",         # misal: harga, pasar, investasi, bank
    2:  "layan, diri, nyata, jernih, luka, sakit, hukum, usia",           # misal: google, ai, digital, startup
    3:  "pro, ai, kamera, ponsel, aplikasi, perangkat, klaim, desain",                 # misal: timnas, gol, liga, pemain
    4:  "film, kisah, tayang, tawa, karakter, diri, usia, cinta"          
}

print("=== INTERPRETASI TOPIK ===")
for topic_id, label in interpretasi.items():
    words = topic_model.get_topic(topic_id)
    kata = [w[0] for w in words[:6]]
    print(f"\nTopik {topic_id:>2} → [{label}]")
    print(f"           Kata kunci: {', '.join(kata)}")

=== INTERPRETASI TOPIK ===

Topik  0 → [ai, ekonomi, teknologi, nilai, rp, persen, sekolah, sistem]
           Kata kunci: ai, ekonomi, teknologi, nilai, rp, persen

Topik  1 → [eropa, menang, jerman, prancis, italia, final, grup, latih]
           Kata kunci: eropa, menang, jerman, prancis, italia, final

Topik  2 → [layan, diri, nyata, jernih, luka, sakit, hukum, usia]
           Kata kunci: layan, diri, nyata, jernih, luka, sakit

Topik  3 → [pro, ai, kamera, ponsel, aplikasi, perangkat, klaim, desain]
           Kata kunci: pro, ai, kamera, ponsel, aplikasi, perangkat

Topik  4 → [film, kisah, tayang, tawa, karakter, diri, usia, cinta]
           Kata kunci: film, kisah, tayang, tawa, karakter, diri


In [18]:
# Simpan info topik ke CSV
topic_info = topic_model.get_topic_info()
topic_info.to_csv("hasil_topik.csv", index=False)

# Simpan mapping dokumen → topik ke CSV
df['topik'] = topics
df['topik_label'] = df['topik'].map(interpretasi).fillna("Outlier / Tidak Terklasifikasi")
df.to_csv("dataset_dengan_topik.csv", index=False)

# Simpan model (opsional, untuk di-load ulang tanpa training lagi)
topic_model.save("model_bertopic_berita")

print("✅ Semua hasil berhasil disimpan:")
print("   - hasil_topik.csv")
print("   - dataset_dengan_topik.csv")
print("   - model_bertopic_berita/")

2026-04-26 12:17:54,455 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


✅ Semua hasil berhasil disimpan:
   - hasil_topik.csv
   - dataset_dengan_topik.csv
   - model_bertopic_berita/
